In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../data/product_type_2.csv", header=[0,1])
df

In [ ]:
### 1. 불필요한 id, product_type, shot 컬럼 제거
df = df.drop(columns=[('process', 'id'), ('process', 'shot'), ('process', 'product_type')])

In [ ]:
### 2. 값이 전부 0인 defects 컬럼 drop
# Defects 하위 컬럼들
defect_cols = df['defects'].columns

# 값이 전부 0인 Defects 컬럼 찾기
zero_defects = [
    col for col in defect_cols
    if df[('defects', col)].sum() == 0
]

# 결과 출력
print("삭제될 defects 컬럼:")
print(zero_defects)

print("\n삭제될 컬럼 수:", len(zero_defects))
print("삭제 전 defects 컬럼 수:", len(defect_cols))

# 실제 삭제
df = df.drop(columns=[('defects', col) for col in zero_defects])

print("삭제 후 defects 컬럼 수:", len(df['defects'].columns))

In [ ]:
df['defects'].sum().sort_values(ascending=False)

In [ ]:
# 불량 범주 정의 (suffix _1, _2 자동 처리)
defect_groups = {
    "표면": [
        "stain_1",
        "dent_1",
        "dent_2",
        "scratch_1",
        "buring_mark_1"
    ],

    "구조": [
        "short_shot_1",
        "short_shot_2",
        "blow_hole_1",
        "blow_hole_2",
        "bubble_1",
        "crack_1",
        "crack_2"
    ],

    "이물질": [
        "contamination_1",
        "contamination_2",
        "impurity_1",
        "impurity_2",
        "inclusions_2"
    ]
}

In [ ]:
import numpy as np
import pandas as pd

defects = df['defects'].copy()  # columns: Short_Shot_1, Blow_Hole_2, ...

# 각 범주별로 해당하는 defect 컬럼들을 찾아서 "하나라도 1이면 1"로 만들기
y_group = pd.DataFrame(index=df.index)

for group_name, base_names in defect_groups.items():
    # base_names 중 하나로 시작하는 컬럼들 찾기 (예: "Short_Shot" -> "Short_Shot_1", "Short_Shot_2")
    cols = [c for c in defects.columns if any(str(c).startswith(b) for b in base_names)]
    
    if len(cols) == 0:
        # 해당 범주에 매칭되는 컬럼이 없으면 0으로
        y_group[group_name] = 0
    else:
        y_group[group_name] = (defects[cols].fillna(0).astype(int).sum(axis=1) > 0).astype(int)

print(y_group.sum().sort_values(ascending=False))   # 범주별 발생 건수 확인
print(y_group.mean().mul(100))                      # 범주별 발생률(%) 확인

In [ ]:
# X = 공정 변수
X = df['process']

# y = 불량 범주
y = y_group

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

xgb = XGBClassifier(
    tree_method="hist",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model = MultiOutputClassifier(xgb)

model.fit(X_train, y_train)

In [ ]:
proba = model.predict_proba(X_test)

import numpy as np
proba_matrix = np.column_stack([p[:, 1] for p in proba])

threshold = 0.3
y_pred = (proba_matrix >= threshold).astype(int)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
print(X_train.shape)

In [ ]:
print(y_group.mean())

In [ ]:
import pandas as pd

importance = model.estimators_[1].feature_importances_

feat_imp = pd.Series(importance, index=X_train.columns)

feat_imp.sort_values(ascending=False).head(10)

In [ ]:
X_train.columns

In [ ]:
y_group.sum(axis=1).value_counts()

In [ ]:
X = df[['Process','Sensor']].copy()

In [ ]:
X.columns = ['_'.join(col) for col in X.columns]

In [ ]:
len(X.columns)

In [ ]:
y = y_group

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_test, y_pred, output_dict=True)

report_df = pd.DataFrame(report).T

label_names = ["구조", "표면", "이물질"]
report_df = report_df.rename(columns={
    "precision": "정밀도",
    "recall": "재현율",
    "f1-score": "F1 점수",
    "support": "샘플 수"
})

print(report_df)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

(y_group.mean()*100).sort_values(ascending=False).plot(kind="bar")
plt.title("Product_Type2 불량 범주 발생률(%)")
plt.ylabel("Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

y_group.sum(axis=1).value_counts().sort_index().plot(kind="bar")
plt.title("Product_Type2 불량 동시 발생 개수 분포")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

df['Defects'].sum().sort_values(ascending=False).head(10).plot(kind="bar")
plt.title("Product_Type2 원본 불량 발생건수 Top10")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
### 3. 남은 defects 중 이상치 확인